# 02a — Supervised Learning: Regression

Predict **continuous numbers** from input features.  
House prices, stock returns, temperature, salary — all regression problems.

```
Input Features ──► Model ──► Continuous Number
  (sq ft, beds)              ($425,000)
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing, make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

---
## 1 — What Is Regression?

Classification asks *"which category?"* — Regression asks *"how much?"*

| Task | Output | Example |
|------|--------|---------|
| Classification | Discrete label | spam / not spam |
| Regression | Continuous value | price = $342,500 |

In [ ]:
np.random.seed(42)
X_demo = 2 * np.random.rand(80, 1)
y_demo = 4 + 3 * X_demo.ravel() + np.random.randn(80) * 0.8

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_demo, y_demo, alpha=0.6, edgecolors='k', linewidths=0.5)
axes[0].set_title('Raw Data — What does the pattern look like?')
axes[0].set_xlabel('Feature (x)')
axes[0].set_ylabel('Target (y)')

m = LinearRegression().fit(X_demo, y_demo)
x_line = np.linspace(0, 2, 100).reshape(-1, 1)
axes[1].scatter(X_demo, y_demo, alpha=0.6, edgecolors='k', linewidths=0.5)
axes[1].plot(x_line, m.predict(x_line), 'r-', linewidth=2, label=f'y = {m.coef_[0]:.2f}x + {m.intercept_:.2f}')
axes[1].set_title('Regression — Fit a line through the data')
axes[1].set_xlabel('Feature (x)')
axes[1].set_ylabel('Target (y)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 2 — Linear Regression

The simplest model: fit a straight line.

### The Math

$$\hat{y} = w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b = \mathbf{w}^T\mathbf{x} + b$$

We want to **minimize** the Mean Squared Error:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Two ways to solve:
- **Normal equation**: $\mathbf{w} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ (closed-form)
- **Gradient descent**: iteratively update weights (we'll implement this)

### 2.1 — Gradient Descent from Scratch

Update rule:

$$w \leftarrow w - \alpha \cdot \frac{\partial \text{MSE}}{\partial w}$$

where $\alpha$ is the learning rate.

In [ ]:
def gradient_descent_linear(X, y, lr=0.1, epochs=100):
    n = len(y)
    w, b = 0.0, 0.0
    history = []

    for epoch in range(epochs):
        y_pred = w * X + b
        error = y_pred - y

        dw = (2 / n) * np.dot(error, X)
        db = (2 / n) * np.sum(error)

        w -= lr * dw
        b -= lr * db

        mse = np.mean(error ** 2)
        history.append(mse)

    return w, b, history

X_flat = X_demo.ravel()
w_gd, b_gd, loss_history = gradient_descent_linear(X_flat, y_demo, lr=0.1, epochs=200)
print(f'From scratch  →  w = {w_gd:.4f}, b = {b_gd:.4f}')
print(f'sklearn       →  w = {m.coef_[0]:.4f}, b = {m.intercept_:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(loss_history)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].set_title('Loss Curve — Gradient Descent Converging')

axes[1].scatter(X_flat, y_demo, alpha=0.5, s=20)
x_range = np.linspace(0, 2, 100)
axes[1].plot(x_range, w_gd * x_range + b_gd, 'r-', linewidth=2, label='From scratch')
axes[1].plot(x_range, m.coef_[0] * x_range + m.intercept_, 'g--', linewidth=2, label='sklearn')
axes[1].set_title('Both solutions match')
axes[1].legend()

plt.tight_layout()
plt.show()

### 2.2 — sklearn LinearRegression

In practice, just use sklearn. It uses the normal equation internally.

In [ ]:
model = LinearRegression()
model.fit(X_demo, y_demo)

print(f'Coefficient (slope):  {model.coef_[0]:.4f}')
print(f'Intercept (bias):     {model.intercept_:.4f}')
print(f'R² score:             {model.score(X_demo, y_demo):.4f}')

---
## 3 — Polynomial Regression

What if the data isn't linear? Add polynomial features: $x \rightarrow [x, x^2, x^3, \ldots]$

Still "linear" regression — linear in the *transformed* features.

In [ ]:
np.random.seed(0)
X_poly = np.sort(np.random.rand(30, 1) * 6, axis=0)
y_poly = np.sin(X_poly).ravel() + np.random.randn(30) * 0.15

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_plot = np.linspace(0, 6, 200).reshape(-1, 1)

for ax, degree, title in zip(axes, [1, 3, 15], 
    ['Degree 1 — UNDERFITTING', 'Degree 3 — GOOD FIT', 'Degree 15 — OVERFITTING']):
    pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    pipe.fit(X_poly, y_poly)
    y_plot = pipe.predict(x_plot)

    ax.scatter(X_poly, y_poly, edgecolors='k', linewidths=0.5, zorder=5)
    ax.plot(x_plot, y_plot, 'r-', linewidth=2)
    ax.plot(x_plot, np.sin(x_plot), 'g--', alpha=0.5, label='True function')
    ax.set_title(title)
    ax.set_ylim(-1.5, 2.0)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Key insight**: higher degree ≠ better model.  
Degree 15 memorizes every point (including noise) but would fail miserably on new data.

---
## 4 — Regularization: Ridge (L2) and Lasso (L1)

Problem: overfitting → weights get huge to chase noise.  
Solution: **penalize large weights** in the loss function.

| Method | Penalty | Effect |
|--------|---------|--------|
| Ridge (L2) | $\lambda \sum w_i^2$ | Shrinks all weights toward zero |
| Lasso (L1) | $\lambda \sum |w_i|$ | Drives some weights exactly to zero (feature selection) |

In [ ]:
alphas = np.logspace(-2, 4, 100)

X_reg, y_reg = make_regression(n_samples=100, n_features=10, noise=20, random_state=42)

ridge_coefs = []
lasso_coefs = []

for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(X_reg, y_reg).coef_)
    lasso_coefs.append(Lasso(alpha=a, max_iter=10000).fit(X_reg, y_reg).coef_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(10):
    axes[0].plot(alphas, [c[i] for c in ridge_coefs], linewidth=1.5)
    axes[1].plot(alphas, [c[i] for c in lasso_coefs], linewidth=1.5)

for ax, title in zip(axes, ['Ridge (L2) — Coefficients shrink', 'Lasso (L1) — Coefficients hit zero']):
    ax.set_xscale('log')
    ax.set_xlabel('Regularization strength (α)')
    ax.set_ylabel('Coefficient value')
    ax.set_title(title)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

**Lasso is doing feature selection** — it zeroes out unimportant features entirely.  
Ridge keeps all features but makes them small.

---
## 5 — Evaluation Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MSE** | $\frac{1}{n}\sum(y - \hat{y})^2$ | Average squared error. Penalizes big errors more. |
| **RMSE** | $\sqrt{\text{MSE}}$ | Same units as y. Most commonly reported. |
| **MAE** | $\frac{1}{n}\sum|y - \hat{y}|$ | Average absolute error. Robust to outliers. |
| **R²** | $1 - \frac{\sum(y - \hat{y})^2}{\sum(y - \bar{y})^2}$ | Fraction of variance explained. 1.0 = perfect. |

In [ ]:
housing = fetch_california_housing()
X_h, y_h = housing.data, housing.target

X_train, X_test, y_train, y_test = train_test_split(X_h, y_h, test_size=0.2, random_state=42)

lr = LinearRegression().fit(X_train, y_train)
y_pred = lr.predict(X_test)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f'MSE:  {mse:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAE:  {mae:.4f}')
print(f'R²:   {r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, y_pred, alpha=0.3, s=10)
axes[0].plot([0, 5], [0, 5], 'r--', linewidth=2)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Linear Regression — R² = {r2:.3f}')

y_random = np.random.permutation(y_test)
r2_bad = r2_score(y_test, y_random)
axes[1].scatter(y_test, y_random, alpha=0.3, s=10, color='orange')
axes[1].plot([0, 5], [0, 5], 'r--', linewidth=2)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title(f'Random "Model" — R² = {r2_bad:.3f}')

plt.tight_layout()
plt.show()

---
## 6 — Train/Test Split: Why You Can't Evaluate on Training Data

A model that memorizes training data looks perfect on training data but fails on new data.  
**Always** evaluate on held-out test data.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

train_r2 = []
test_r2 = []
degrees = range(1, 6)

for d in degrees:
    poly_pipe = make_pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression())
    poly_pipe.fit(X_train_s[:500], y_train[:500])
    train_r2.append(poly_pipe.score(X_train_s[:500], y_train[:500]))
    test_r2.append(poly_pipe.score(X_test_s, y_test))

plt.figure(figsize=(8, 5))
plt.plot(list(degrees), train_r2, 'bo-', label='Train R²', linewidth=2)
plt.plot(list(degrees), test_r2, 'rs-', label='Test R²', linewidth=2)
plt.xlabel('Polynomial Degree')
plt.ylabel('R² Score')
plt.title('Overfitting: Train score keeps rising, test score drops')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

The gap between train and test performance is the **hallmark of overfitting**.  
If train R² is much higher than test R² → your model is memorizing, not learning.

---
## 7 — End-to-End Example: California Housing

Full pipeline: load → explore → preprocess → train → evaluate → visualize.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(f'Shape: {df.shape}')
print(f'Target: {housing.target_names}')
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.ravel(), housing.feature_names):
    ax.scatter(df[col], df['MedHouseVal'], alpha=0.1, s=2)
    ax.set_xlabel(col)
    ax.set_ylabel('MedHouseVal')
plt.suptitle('Feature vs Target — Looking for patterns', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)':     Ridge(alpha=1.0),
    'Lasso (α=0.01)':    Lasso(alpha=0.01),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    results[name] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE':  mean_absolute_error(y_test, y_pred),
        'R²':   r2_score(y_test, y_pred),
    }

pd.DataFrame(results).T.round(4)

In [ ]:
best_model = models['Random Forest']
y_pred_best = best_model.predict(X_test_s)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_best, alpha=0.2, s=5)
axes[0].plot([0, 5], [0, 5], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Price ($100k)')
axes[0].set_ylabel('Predicted Price ($100k)')
axes[0].set_title(f'Random Forest — R² = {results["Random Forest"]["R²"]:.3f}')

importances = best_model.feature_importances_
idx = np.argsort(importances)
axes[1].barh(range(len(idx)), importances[idx])
axes[1].set_yticks(range(len(idx)))
axes[1].set_yticklabels(np.array(housing.feature_names)[idx])
axes[1].set_title('Feature Importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_pred_best, residuals, alpha=0.2, s=5)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residual Plot — should be random around 0')

axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution — should be roughly normal')

plt.tight_layout()
plt.show()

---
## Key Takeaways

| Concept | Remember |
|---------|----------|
| Linear Regression | Simplest baseline — always start here |
| Polynomial Regression | More flexible, but watch for overfitting |
| Ridge / Lasso | Regularization prevents overfitting. Lasso does feature selection. |
| Train/Test Split | Never evaluate on training data |
| RMSE vs MAE | RMSE penalizes big errors more; MAE is more robust |
| R² | 1.0 = perfect, 0.0 = predicts the mean, negative = worse than mean |

**Next**: Classification — when the target is a category, not a number →